# 06B_MODIS_Feature_Extraction_v2.ipynb

Improved MODIS extraction.

Features:
- Upload `image_metadata.csv`
- Uses `MODIS/061/MOD11A1`
- Progressive search: ±3 → ±7 → ±15 days
- Uses a 500 m buffer around each event
- Records the actual MODIS image date used
- Outputs:
  - `modis_features.csv`
  - `modis_skipped.csv`


In [ ]:
!pip -q install earthengine-api pandas tqdm

import ee
import pandas as pd
from tqdm import tqdm
from google.colab import files

ee.Authenticate()
ee.Initialize(project="heatwave-flood-prediction")

print("Upload image_metadata.csv")
uploaded = files.upload()
fname = list(uploaded.keys())[0]

df = pd.read_csv(fname)

results = []
skipped = []

windows = [3, 7, 15]

for _, row in tqdm(df.iterrows(), total=len(df)):

    event_id = row["event_id"]
    img_date = ee.Date(str(row["image_date"]))
    point = ee.Geometry.Point([float(row["longitude"]), float(row["latitude"])])
    region = point.buffer(500)

    found = False

    for days in windows:

        collection = (
            ee.ImageCollection("MODIS/061/MOD11A1")
            .filterBounds(region)
            .filterDate(img_date.advance(-days, "day"),
                        img_date.advance(days + 1, "day"))
            .sort("system:time_start")
        )

        size = collection.size().getInfo()

        if size == 0:
            continue

        imgs = collection.toList(size)

        for i in range(size):

            image = ee.Image(imgs.get(i))

            vals = image.reduceRegion(
                reducer=ee.Reducer.mean(),
                geometry=region,
                scale=1000,
                maxPixels=1e9
            ).getInfo()

            day = vals.get("LST_Day_1km")
            night = vals.get("LST_Night_1km")

            if day is None and night is None:
                continue

            modis_date = ee.Date(image.get("system:time_start")).format("YYYY-MM-dd").getInfo()

            results.append({
                "event_id": event_id,
                "image_date": row["image_date"],
                "modis_date": modis_date,
                "search_window_days": days,
                "LST_Day_C": None if day is None else day * 0.02 - 273.15,
                "LST_Night_C": None if night is None else night * 0.02 - 273.15
            })

            found = True
            break

        if found:
            break

    if not found:
        skipped.append({
            "event_id": event_id,
            "image_date": row["image_date"],
            "reason": "No valid MODIS LST within ±15 days"
        })

pd.DataFrame(results).to_csv("modis_features.csv", index=False)
pd.DataFrame(skipped).to_csv("modis_skipped.csv", index=False)

print("Completed")
print("Extracted:", len(results))
print("Skipped:", len(skipped))

files.download("modis_features.csv")
files.download("modis_skipped.csv")
